# DeepPrep on FABRIC — SUDMEX_CONN Pipeline (Final)Full pipeline: neuroimaging preprocessing + FC extraction + ML classification, distributed across 3 FABRIC VMs.## What this notebook does1. **Cell 1** — Initialize fablib and verify credentials2. **Cell 2** — Nuclear reset (deletes any existing DeepPrep slice for a clean start)3. **Cell 3** — Configuration (site, nodes, subjects, paths, FreeSurfer license)4. **Cell 4** — Create FABRIC slice with 3 nodes (32 cores, 64GB RAM each at GATECH/NCSA)5. **Cell 5** — Wait for nodes to become SSH-ready6. **Cell 6** — Verify node resources (CPU/RAM/disk)7. **Cell 7** — Install Docker on all 3 nodes in parallel8. **Cell 8** — Download BIDS data + install openneuro-py in parallel9. **Cell 9** — Pull DeepPrep Docker image (~15GB)10. **Cell 10** — Run DeepPrep preprocessing in parallel (per-subject on each node)11. **Cell 11** — Verify DeepPrep outputs exist per node12. **Cell 12** — Copy preprocessed BOLD outputs to JupyterHub13. **Cell 13** — Install nilearn + ML deps on JupyterHub14. **Cell 14-17** — Legacy JupyterHub-based FC extraction (replaced by Cells 20-22 due to memory constraints)15. **Cell 18** — Bundle FC matrices + participants.tsv into a zip16. **Cell 19** — Delete slice (saves FABRIC credits)## NEW CELLS (what we actually used)Because JupyterHub kept running out of memory when loading preprocessed BOLD files, we moved FC extraction directly onto the FABRIC VMs (which have 64GB RAM each):17. **Cell 20** — Upload FC extraction script to each FABRIC VM18. **Cell 21** — Run FC extraction in parallel on all VMs19. **Cell 22** — Download FC matrices back to JupyterHub20. **Cell 23** — Batch processing for additional subjects21. **Cell 24** — Create final classifier zip ready for ML pipeline## Team- **Connor Eltoft** — FABRIC slice + DeepPrep notebook orchestration- **Jack Frater** — FC extraction troubleshooting, ML classifier, visualization- **Moses Osineye** — Report + multi-dataset integration## Datasets- **SUDMEX_CONN (ds003346)** — Cocaine Use Disorder cohort with healthy controls (145 subjects, used 4 for this demo)

## Cell 1 — Initialize fablib

In [ ]:
# Cell 1: Initialize FABRIC fablib library# Connects to your FABRIC credentials and verifies the bastion SSH key.# Must be run first — all other cells depend on `fablib` being available.try:    from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib    fablib = fablib()    print('✅ fablib ready!')except Exception as e:    print(f'❌ Failed to initialize fablib: {e}')

In [13]:
import time

# ============================================================
# AUTOMATED SITE FINDER — tries each site until one works
# ============================================================
SLICE_NAME = 'DeepPrep'
NUM_NODES  = 3
NODE_NAMES = [f'DeepPrepNode{i+1}' for i in range(NUM_NODES)]
NODE_DISK  = 100

FS_LICENSE_CONTENT = """buyol17@gmail.com
94989
 *CQiGbN3QRd0g
 FSJv4ryaE59RA
 8REXhZofoHYWEKxbWsA5fbAxg4ewJiGzuKck/yi/HfU="""

# Phase 1: try remaining sites at 32 cores / 64 GB
phase1_sites = ['NCSA', 'MAX', 'GPN', 'EDUKY']
# Phase 2 (if phase 1 all fail): all sites at 16 cores / 32 GB
phase2_sites = ['NCSA', 'MAX', 'GPN', 'EDUKY', 'GATECH', 'UCSD', 'TACC']

def try_delete_slice():
    try:
        s = fablib.get_slice(SLICE_NAME)
        print(f'  Deleting existing slice...')
        s.delete()
        time.sleep(5)
        print(f'  ✅ Deleted.')
    except Exception as e:
        if 'not found' in str(e).lower() or 'does not exist' in str(e).lower():
            print(f'  No existing slice to delete.')
        else:
            print(f'  Warning during delete: {str(e)[:80]}')

def try_site(site, cores, ram):
    print(f'\n{"="*60}')
    print(f'Trying {site} | {cores} cores | {ram}GB RAM ...')
    print(f'{"="*60}')
    
    # Step 1: delete any existing slice
    try_delete_slice()
    time.sleep(5)  # brief pause
    
    # Step 2: create slice
    try:
        slice_obj = fablib.new_slice(name=SLICE_NAME)
        for node_name in NODE_NAMES:
            node = slice_obj.add_node(
                name=node_name, site=site,
                cores=cores, ram=ram, disk=NODE_DISK,
                image='default_ubuntu_22'
            )
            net   = slice_obj.add_l2network(name=f'net_{node_name}')
            iface = node.add_component(model='NIC_Basic', name=f'nic_{node_name}').get_interfaces()[0]
            net.add_interface(iface)
        
        slice_obj.submit(wait=False)
        print(f'  Slice submitted, waiting for result...')
        
        # Poll for up to 120 seconds
        for attempt in range(24):
            time.sleep(5)
            try:
                s = fablib.get_slice(SLICE_NAME)
                state = s.get_state()
                print(f'  [{attempt*5+5}s] State: {state}')
                if state == 'StableOK':
                    print(f'  ✅ SUCCESS! {site} with {cores} cores / {ram}GB RAM is READY!')
                    return True, site, cores, ram, s
                elif state in ('Dead', 'StableError', 'Closing', 'Closed'):
                    # Check node errors
                    try:
                        nodes = s.get_nodes()
                        for n in nodes:
                            err = n.get_error()
                            if err:
                                print(f'  Node {n.get_name()} error: {str(err)[:80]}')
                    except:
                        pass
                    print(f'  ❌ FAILED at {site}: state={state}')
                    return False, site, cores, ram, None
            except Exception as poll_err:
                print(f'  Poll error: {str(poll_err)[:60]}')
        
        print(f'  ❌ TIMEOUT at {site} after 120s')
        return False, site, cores, ram, None
        
    except Exception as e:
        print(f'  ❌ Submit error at {site}: {str(e)[:100]}')
        return False, site, cores, ram, None

# ---- PHASE 1: 32 cores / 64 GB ----
print('\n🔍 PHASE 1: Trying sites at 32 cores / 64GB RAM')
winning_site = None
winning_cores = None
winning_ram = None
winning_slice = None

for site in phase1_sites:
    success, s, c, r, slc = try_site(site, 32, 64)
    if success:
        winning_site, winning_cores, winning_ram, winning_slice = s, c, r, slc
        break

# ---- PHASE 2: 16 cores / 32 GB (if phase 1 all failed) ----
if not winning_site:
    print('\n⚠️  All Phase 1 sites failed. Switching to 16 cores / 32GB RAM...')
    for site in phase2_sites:
        success, s, c, r, slc = try_site(site, 16, 32)
        if success:
            winning_site, winning_cores, winning_ram, winning_slice = s, c, r, slc
            break

if winning_site:
    # Store globals for subsequent cells
    SITE       = winning_site
    NODE_CORES = winning_cores
    NODE_RAM   = winning_ram
    slice      = winning_slice
    nodes      = {name: slice.get_node(name) for name in NODE_NAMES}
    print(f'\n🎉 WINNER: {winning_site} | {winning_cores} cores | {winning_ram}GB RAM')
    print(f'   Slice state: {slice.get_state()}')
else:
    print('\n💀 ALL SITES FAILED at both 32 and 16 cores. FABRIC may be overloaded. Try again later.')



🔍 PHASE 1: Trying sites at 32 cores / 64GB RAM

Trying NCSA | 32 cores | 64GB RAM ...
  Deleting existing slice...
  ✅ Deleted.
  Slice submitted, waiting for result...
  [5s] State: Configuring
  [10s] State: Configuring
  [15s] State: Configuring
  [20s] State: Configuring
  [25s] State: Configuring
  [30s] State: Configuring
  [35s] State: Configuring
  [40s] State: Configuring
  [45s] State: Configuring
  [50s] State: Configuring
  [55s] State: Configuring
  [60s] State: Configuring
  [65s] State: Configuring
  [70s] State: Configuring
  [75s] State: Configuring
  [80s] State: Configuring
  [85s] State: Configuring
  [90s] State: Configuring
  [95s] State: Configuring
  [100s] State: Configuring
  [105s] State: Configuring
  [110s] State: Configuring
  [115s] State: Configuring
  [120s] State: Configuring
  ❌ TIMEOUT at NCSA after 120s

Trying MAX | 32 cores | 64GB RAM ...
  Deleting existing slice...
  Warning during delete: 500: HTTP request failed
{
    "errors": [
        {
  

## Cell 2 — Nuclear Reset (delete existing slice if any)

In [7]:
# Run this to wipe any existing DeepPrep slice before starting fresh.
# Safe to run even if no slice exists.

try:
    existing = fablib.get_slice('DeepPrep')
    print('Found existing slice — deleting...')
    existing.delete()
    print('✅ Existing slice deleted. Wait ~30 seconds before running Cell 3.')
except Exception as e:
    if 'not found' in str(e).lower() or 'does not exist' in str(e).lower():
        print('✅ No existing slice found — good to go.')
    else:
        print(f'⚠️  Unexpected error: {e}')

Found existing slice — deleting...
✅ Existing slice deleted. Wait ~30 seconds before running Cell 3.


## Cell 3 — Configuration

In [2]:
import os

# --- FABRIC slice/node names ---
SLICE_NAME = 'DeepPrep'
SITE       = 'NCSA'

# Number of parallel nodes
NUM_NODES = 3

# One node name per worker
NODE_NAMES = [f'DeepPrepNode{i+1}' for i in range(NUM_NODES)]

# --- Node specs (per node — each handles 1 subject) ---
# Bumped to 32 cores and 64GB RAM for significantly faster DeepPrep runtime
NODE_CORES = 32
NODE_RAM   = 64
NODE_DISK  = 100

# --- Dataset: 3 subjects, one assigned to each node ---
DATASET_ID = 'ds003346'
SUBJECTS   = ['001', '002', '003']

# Map subject -> node name
SUBJECT_NODE_MAP = {sub: NODE_NAMES[i] for i, sub in enumerate(SUBJECTS)}

# --- DeepPrep ---
DEEPPREP_VERSION  = '25.1.0'
DEEPPREP_IMAGE    = f'pbfslab/deepprep:{DEEPPREP_VERSION}'
BOLD_TASK_TYPE    = 'rest'
BOLD_VOLUME_SPACE = 'MNI152NLin2009cAsym'

# --- Local paths (JupyterHub) ---
LOCAL_BASE       = os.path.join(os.path.expanduser('~'), 'sudmex')
LOCAL_BIDS_DIR   = os.path.join(LOCAL_BASE, 'bids')
LOCAL_OUTPUT_DIR = os.path.join(LOCAL_BASE, 'deepprep_output')
LOCAL_FC_DIR     = os.path.join(LOCAL_BASE, 'fc_matrices')

for path in [LOCAL_BASE, LOCAL_BIDS_DIR, LOCAL_OUTPUT_DIR, LOCAL_FC_DIR]:
    os.makedirs(path, exist_ok=True)

# --- Remote paths (same layout on every node) ---
REMOTE_BASE       = '/home/ubuntu/sudmex'
REMOTE_BIDS_DIR   = f'{REMOTE_BASE}/bids'
REMOTE_OUTPUT_DIR = f'{REMOTE_BASE}/deepprep_output'
REMOTE_LICENSE    = f'{REMOTE_BASE}/fs_license.txt'

# --- FreeSurfer license ---
# Paste your real license content here (free from https://surfer.nmr.mgh.harvard.edu/registration.html)
FS_LICENSE_CONTENT = """buyol17@gmail.com
94989
 *CQiGbN3QRd0g
 FSJv4ryaE59RA
 8REXhZofoHYWEKxbWsA5fbAxg4ewJiGzuKck/yi/HfU="""

print('✅ Config ready')
print(f'  Slice:    {SLICE_NAME} @ {SITE}')
print(f'  Nodes:    {NODE_NAMES}')
print(f'  Cores/node: {NODE_CORES}  RAM/node: {NODE_RAM}GB')
print(f'  Subjects: {SUBJECTS}')
print(f'  Subject→Node map: {SUBJECT_NODE_MAP}')
print(f'  Local out: {LOCAL_OUTPUT_DIR}')

✅ Config ready
  Slice:    DeepPrep @ NCSA
  Nodes:    ['DeepPrepNode1', 'DeepPrepNode2', 'DeepPrepNode3']
  Cores/node: 32  RAM/node: 64GB
  Subjects: ['001', '002', '003']
  Subject→Node map: {'001': 'DeepPrepNode1', '002': 'DeepPrepNode2', '003': 'DeepPrepNode3'}
  Local out: /home/fabric/sudmex/deepprep_output


## Cell 4 — Create FABRIC slice

In [6]:
try:
    print('Creating slice with 3 nodes...')
    slice = fablib.new_slice(name=SLICE_NAME)

    for node_name in NODE_NAMES:
        node = slice.add_node(
            name=node_name,
            site=SITE,
            cores=NODE_CORES,
            ram=NODE_RAM,
            disk=NODE_DISK,
            image='default_ubuntu_22'
        )
        net   = slice.add_l2network(name=f'net_{node_name}')
        iface = node.add_component(model='NIC_Basic', name=f'nic_{node_name}').get_interfaces()[0]
        net.add_interface(iface)
        print(f'  ✅ {node_name} added ({NODE_CORES} cores, {NODE_RAM}GB RAM, {NODE_DISK}GB disk)')

    slice.submit()
    print('\n✅ Slice submitted! Run next cell to wait for all nodes to become ready.')
except Exception as e:
    print(f'❌ Error: {e}')


Retry: 1, Time: 41 sec


ID,e2f87193-9d09-4b11-b99d-31762be104e8
Name,DeepPrep
Lease Expiration (UTC),2026-04-24 04:33:21 +0000
Lease Start (UTC),2026-04-23 04:33:21 +0000
Project ID,7db31408-b695-427c-8856-8761292228b6
State,Dead
Email,fkb8qp@health.missouri.edu
UserId,86c661f5-4559-474b-979a-96612f26c177


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
b7fa004c-fee6-4f2c-91b1-325b5f0a8b25,DeepPrepNode1,0,0,0,default_ubuntu_22,qcow2,None,GATECH,ubuntu,,Closed,Insufficient resources : [core]#,,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
649cd0f8-fb3f-4de9-840a-e8091747e929,DeepPrepNode2,0,0,0,default_ubuntu_22,qcow2,None,GATECH,ubuntu,,Closed,Insufficient resources : [core]#,,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
24e8f7ce-128c-4df7-ab52-89ff9033dfbb,DeepPrepNode3,0,0,0,default_ubuntu_22,qcow2,None,GATECH,ubuntu,,Closed,Insufficient resources : [core]#,,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
d4882121-f2fa-4036-aa3c-6c7d7e1ffb31,net_DeepPrepNode1,L2,L2Bridge,GATECH,net_DeepPrepNode1.gateway,net_DeepPrepNode1.subnet,Closed,"redeem predecessor reservation# b7fa004c-fee6-4f2c-91b1-325b5f0a8b25 is in a terminal state, failing the reservation# d4882121-f2fa-4036-aa3c-6c7d7e1ffb31#"
da7af0e6-2e34-40ab-a954-31597dc1c709,net_DeepPrepNode2,L2,L2Bridge,GATECH,net_DeepPrepNode2.gateway,net_DeepPrepNode2.subnet,Closed,"redeem predecessor reservation# 649cd0f8-fb3f-4de9-840a-e8091747e929 is in a terminal state, failing the reservation# da7af0e6-2e34-40ab-a954-31597dc1c709#"
68f26168-dbe9-43f7-a12d-5b8c8f69315c,net_DeepPrepNode3,L2,L2Bridge,GATECH,net_DeepPrepNode3.gateway,net_DeepPrepNode3.subnet,Closed,"redeem predecessor reservation# 24e8f7ce-128c-4df7-ab52-89ff9033dfbb is in a terminal state, failing the reservation# 68f26168-dbe9-43f7-a12d-5b8c8f69315c#"


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
DeepPrepNode1-nic_DeepPrepNode1-p1,p1,DeepPrepNode1,net_DeepPrepNode1,100,,,,,config,,,p1
DeepPrepNode2-nic_DeepPrepNode2-p1,p1,DeepPrepNode2,net_DeepPrepNode2,100,,,,,config,,,p1
DeepPrepNode3-nic_DeepPrepNode3-p1,p1,DeepPrepNode3,net_DeepPrepNode3,100,,,,,config,,,p1



Time to print interfaces 41 seconds

✅ Slice submitted! Run next cell to wait for all nodes to become ready.


## Cell 5 — Wait for nodes to become active (3-5 mins)

In [5]:
try:
    print('Waiting for SSH on all 3 nodes... (this can take 3-5 minutes)')
    slice.wait_ssh(progress=True)
    print('✅ All nodes are ready!')
    print(slice.list_nodes())
except Exception as e:
    print(f'❌ Failed waiting for nodes: {e}')

Waiting for SSH on all 3 nodes... (this can take 3-5 minutes)
Waiting for slice . Slice state: StableOK
Waiting for ssh in slice . ssh successful
✅ All nodes are ready!


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
30e65aa5-fbf7-4562-8983-b92e20fe3410,DeepPrepNode1,32,64,100,default_ubuntu_22,qcow2,gatech-w3.fabric-testbed.net,GATECH,ubuntu,2610:148:1f00:9f01:f816:3eff:fe3a:7a36,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:148:1f00:9f01:f816:3eff:fe3a:7a36,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
10b47434-53e1-4dc0-a300-633e370617c3,DeepPrepNode2,32,64,100,default_ubuntu_22,qcow2,gatech-w3.fabric-testbed.net,GATECH,ubuntu,2610:148:1f00:9f01:f816:3eff:fe83:7425,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:148:1f00:9f01:f816:3eff:fe83:7425,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
8f7db8f5-56b5-4530-b469-1799efcd9583,DeepPrepNode3,32,64,100,default_ubuntu_22,qcow2,gatech-w3.fabric-testbed.net,GATECH,ubuntu,2610:148:1f00:9f01:f816:3eff:fe4c:b088,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:148:1f00:9f01:f816:3eff:fe4c:b088,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


## Cell 6 — Get node handles and verify resources

In [6]:
try:
    slice = fablib.get_slice(SLICE_NAME)
    nodes = {name: slice.get_node(name) for name in NODE_NAMES}

    print('✅ All nodes found!')
    for name, node in nodes.items():
        sub = [s for s, n in SUBJECT_NODE_MAP.items() if n == name][0]
        stdout, _ = node.execute('nproc && free -h && df -h /')
        print(f'\n--- {name} (sub-{sub}) ---')
        print(stdout)
except Exception as e:
    print(f'❌ Failed to get node info: {e}')

✅ All nodes found!
32
               total        used        free      shared  buff/cache   available
Mem:            62Gi       356Mi        62Gi       4.0Mi       290Mi        61Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
/dev/vda1        97G  1.6G   96G   2% /

--- DeepPrepNode1 (sub-001) ---
32
               total        used        free      shared  buff/cache   available
Mem:            62Gi       356Mi        62Gi       4.0Mi       290Mi        61Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
/dev/vda1        97G  1.6G   96G   2% /

32
               total        used        free      shared  buff/cache   available
Mem:            62Gi       358Mi        62Gi       4.0Mi       299Mi        61Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
/dev/vda1        97G  1.6G   96G   2% /

--- DeepPrepNode2 (sub-002) ---
32
              

## Cell 7 — Install Docker on all nodes (parallel)

In [14]:
import threading

def install_docker(node, node_name):
    try:
        node.execute('sudo apt-get update -y')
        node.execute(
            'sudo apt-get install -y ca-certificates curl && '
            'sudo install -m 0755 -d /etc/apt/keyrings && '
            'sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc && '
            'sudo chmod a+r /etc/apt/keyrings/docker.asc && '
            'echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] '
            'https://download.docker.com/linux/ubuntu $(. /etc/os-release && echo $VERSION_CODENAME) stable" | '
            'sudo tee /etc/apt/sources.list.d/docker.list > /dev/null && '
            'sudo apt-get update -y && '
            'sudo apt-get install -y docker-ce docker-ce-cli containerd.io'
        )
        node.execute('sudo usermod -aG docker ubuntu && newgrp docker')
        stdout, _ = node.execute('docker --version')
        print(f'  ✅ {node_name}: {stdout.strip()}')
    except Exception as e:
        print(f'  ❌ {node_name}: Docker install failed — {e}')

try:
    slice = fablib.get_slice(SLICE_NAME)
    nodes = {name: slice.get_node(name) for name in NODE_NAMES}

    print('Installing Docker on all 3 nodes in parallel...')
    threads = []
    for name, node in nodes.items():
        t = threading.Thread(target=install_docker, args=(node, name))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print('\n✅ Docker installed on all nodes.')
except Exception as e:
    print(f'❌ Error: {e}')

Installing Docker on all 3 nodes in parallel...
Hit:1 http://nova.clouds.archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://nova.clouds.archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://nova.clouds.archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://nova.clouds.archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 http://nova.clouds.archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:1 http://nova.clouds.archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3170 kB]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3170 kB]
Get:6 http://nova.clouds.archive.ubuntu.com/ubuntu jammy/universe amd64 Packages [14.1 MB]
Get:3 http://n

## Cell 8 — Install deps and download BIDS data (parallel)

In [15]:
import threading

def setup_node(node, node_name, subject):
    try:
        node.execute('sudo apt-get install -y python3-pip')
        node.execute('sudo pip3 install openneuro-py')

        node.execute(f'mkdir -p {REMOTE_BIDS_DIR} {REMOTE_OUTPUT_DIR}')

        license_escaped = FS_LICENSE_CONTENT.replace('"', '\\"')
        node.execute(f'echo "{license_escaped}" > {REMOTE_LICENSE}')

        subject_includes = str([
            f'sub-{subject}',
            'dataset_description.json',
            'participants.tsv',
            'participants.json',
            'README.md',
        ])

        download_script = (
            'import openneuro\n'
            f'include_list = {subject_includes}\n'
            f'openneuro.download(\n'
            f'    dataset="ds003346",\n'
            f'    target_dir="{REMOTE_BIDS_DIR}",\n'
            f'    include=include_list,\n'
            f')\n'
            'print("Download complete.")\n'
        )

        node.execute(
            f"cat > /home/ubuntu/download_bids.py << 'PYEOF'\n{download_script}\nPYEOF"
        )

        stdout, stderr = node.execute('sudo python3 /home/ubuntu/download_bids.py', timeout=600)
        if 'Download complete' in stdout:
            print(f'  ✅ {node_name} (sub-{subject}): BIDS download complete')
        else:
            print(f'  ❌ {node_name} (sub-{subject}): download may have failed')
            print(f'     stdout: {stdout[:300]}')
            print(f'     stderr: {stderr[:300]}')
    except Exception as e:
        print(f'  ❌ {node_name} (sub-{subject}): FAILED — {e}')

try:
    slice = fablib.get_slice(SLICE_NAME)
    nodes = {name: slice.get_node(name) for name in NODE_NAMES}

    print('Setting up all 3 nodes in parallel (deps + BIDS download)...')
    threads = []
    for sub, node_name in SUBJECT_NODE_MAP.items():
        node = nodes[node_name]
        t = threading.Thread(target=setup_node, args=(node, node_name, sub))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print('\n✅ All nodes set up and data downloaded.')
except Exception as e:
    print(f'❌ Error: {e}')

Setting up all 3 nodes in parallel (deps + BIDS download)...
Reading package lists...Reading package lists...
Building dependency tree...
Building dependency tree...Reading package lists...
Building dependency tree...
Reading state information...

Reading state information...
The following additional packages will be installed:
  build-essential bzip2 cpp cpp-11 dpkg-dev fakeroot fontconfig-config
  fonts-dejavu-core g++ g++-11 gcc gcc-11 gcc-11-base gcc-12-base
  javascript-common libalgorithm-diff-perl libalgorithm-diff-xs-perl
  libalgorithm-merge-perl libasan6 libatomic1 libc-dev-bin libc-devtools libc6
  libc6-dev libcc1-0 libcrypt-dev libdeflate0 libdpkg-perl libexpat1
  libexpat1-dev libfakeroot libfile-fcntllock-perl libfontconfig1
  libgcc-11-dev libgcc-s1 libgd3 libgomp1 libisl23 libitm1 libjbig0
  libjpeg-turbo8 libjpeg8 libjs-jquery libjs-sphinxdoc libjs-underscore
  liblsan0 libmpc3 libnsl-dev libpython3-dev libpython3.10 libpython3.10-dev
  libpython3.10-minimal libpython

## Cell 9 — Pull DeepPrep Docker image (parallel)

In [16]:
import threading

def pull_image(node, node_name):
    try:
        print(f'  ⏳ {node_name}: pulling {DEEPPREP_IMAGE}...')
        node.execute(f'sudo docker pull {DEEPPREP_IMAGE}', timeout=1800)
        print(f'  ✅ {node_name}: image pulled')
    except Exception as e:
        print(f'  ❌ {node_name}: pull failed — {e}')

try:
    slice = fablib.get_slice(SLICE_NAME)
    nodes = {name: slice.get_node(name) for name in NODE_NAMES}

    print(f'Pulling {DEEPPREP_IMAGE} on all 3 nodes in parallel...')
    print('Large image (~10-15GB) — expect 10-20 minutes.')
    threads = []
    for name, node in nodes.items():
        t = threading.Thread(target=pull_image, args=(node, name))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print('\n✅ DeepPrep image ready on all nodes.')
except Exception as e:
    print(f'❌ Error: {e}')

Pulling pbfslab/deepprep:25.1.0 on all 3 nodes in parallel...
Large image (~10-15GB) — expect 10-20 minutes.
  ⏳ DeepPrepNode1: pulling pbfslab/deepprep:25.1.0...
  ⏳ DeepPrepNode2: pulling pbfslab/deepprep:25.1.0...
  ⏳ DeepPrepNode3: pulling pbfslab/deepprep:25.1.0...
25.1.0: Pulling from pbfslab/deepprep
25.1.0: Pulling from pbfslab/deepprep
25.1.0: Pulling from pbfslab/deepprep
5e3b7ee77381: Pulling fs layer
eeec3542c7d6: Pulling fs layer
507fc9045cba: Pulling fs layer
263fc748118f: Pulling fs layer
55b3657547f8: Pulling fs layer
43f89b94cd7d: Pulling fs layer
0cb1ef9ad21e: Pulling fs layer
16c36d0187d0: Pulling fs layer
6f73eeea6237: Pulling fs layer
340cf11aa5ff: Pulling fs layer
8d863280327f: Pulling fs layer
8c013fcce11d: Pulling fs layer
d377a39e6b5c: Pulling fs layer
74da4e43c253: Pulling fs layer
47d45bcfa5d8: Pulling fs layer
61013afefe0d: Pulling fs layer
ebac9623d7df: Pulling fs layer
64e62cf848cc: Pulling fs layer
d18238905db2: Pulling fs layer
de4e1a4960d2: Pulling fs l

## Cell 10 — Run DeepPrep on all nodes (parallel, silent polling)

In [22]:
import threading
import time

def run_deepprep(node, node_name, subject):
    try:
        deepprep_cmd = (
            f'sudo docker run --rm '
            f'-v {REMOTE_BIDS_DIR}:/input '
            f'-v {REMOTE_OUTPUT_DIR}:/output '
            f'-v {REMOTE_LICENSE}:/fs_license.txt '
            f'{DEEPPREP_IMAGE} '
            f'/input /output participant '
            f'--bold_task_type {BOLD_TASK_TYPE} '
            f'--fs_license_file /fs_license.txt '
            f'--participant_label sub-{subject} '   # wrapped in single quotes
            f'--bold_confounds '
            f'--bold_volume_space {BOLD_VOLUME_SPACE} '
            f'--bold_surface_spaces None '
            f'--bold_skip_frame 2 '
            f'--device cpu '
            f'--cpus {NODE_CORES} '
            f'--memory {NODE_RAM} '
            f'--ignore_error'
        )

        log_path = f'{REMOTE_BASE}/deepprep_run.log'
        background_cmd = f'nohup bash -c "{deepprep_cmd} > {log_path} 2>&1" &'

        node.execute(background_cmd)
        print(f'  ✅ {node_name} (sub-{subject}): DeepPrep launched')

        # Poll silently every 60 seconds — no log spam
        start_time = time.time()
        while True:
            stdout_ps, _ = node.execute('pgrep -f "docker run" | wc -l')
            if int(stdout_ps.strip()) == 0:
                elapsed = int(time.time() - start_time)
                print(f'  ✅ {node_name} (sub-{subject}): DONE in {elapsed//60}m {elapsed%60}s')
                break
            elapsed = int(time.time() - start_time)
            print(f'  ⏳ {node_name} (sub-{subject}): still running... {elapsed//60}m {elapsed%60}s elapsed', flush=True)
            time.sleep(60)

    except Exception as e:
        print(f'  ❌ {node_name} (sub-{subject}): ERROR — {e}')

try:
    slice = fablib.get_slice(SLICE_NAME)
    nodes = {name: slice.get_node(name) for name in NODE_NAMES}

    print('Running DeepPrep on all 3 nodes simultaneously...')
    threads = []
    for sub, node_name in SUBJECT_NODE_MAP.items():
        node = nodes[node_name]
        t = threading.Thread(target=run_deepprep, args=(node, node_name, sub))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print('\n✅ DeepPrep finished on all nodes!')
except Exception as e:
    print(f'❌ Error: {e}')

Running DeepPrep on all 3 nodes simultaneously...
  ✅ DeepPrepNode1 (sub-001): DeepPrep launched
4
  ⏳ DeepPrepNode1 (sub-001): still running... 0m 1s elapsed
  ✅ DeepPrepNode3 (sub-003): DeepPrep launched
1
  ⏳ DeepPrepNode3 (sub-003): still running... 0m 0s elapsed
4
  ⏳ DeepPrepNode1 (sub-001): still running... 1m 3s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 1m 0s elapsed
4
  ⏳ DeepPrepNode1 (sub-001): still running... 2m 3s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 2m 0s elapsed
4
  ⏳ DeepPrepNode1 (sub-001): still running... 3m 3s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 3m 0s elapsed
4
  ⏳ DeepPrepNode1 (sub-001): still running... 4m 3s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 4m 0s elapsed
4
  ⏳ DeepPrepNode1 (sub-001): still running... 5m 3s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 5m 0s elapsed
4
  ⏳ DeepPrepNode1 (sub-001): still running... 6m 3s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 6m

KeyboardInterrupt: 

1
1
1
  ⏳ DeepPrepNode2 (sub-002): still running... 13m 1s elapsed
1
  ❌ DeepPrepNode1 (sub-001): ERROR — Socket operation on non-socket
1
  ⏳ DeepPrepNode3 (sub-003): still running... 106m 9s elapsed
1
  ⏳ DeepPrepNode2 (sub-002): still running... 14m 1s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 107m 10s elapsed
1
  ⏳ DeepPrepNode2 (sub-002): still running... 15m 1s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 108m 10s elapsed
1
  ⏳ DeepPrepNode2 (sub-002): still running... 16m 1s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 109m 10s elapsed
1
  ⏳ DeepPrepNode2 (sub-002): still running... 17m 1s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 110m 10s elapsed
1
  ⏳ DeepPrepNode2 (sub-002): still running... 18m 1s elapsed
1
  ⏳ DeepPrepNode3 (sub-003): still running... 111m 10s elapsed


## Cell 11 — Verify outputs on nodes

In [23]:
try:
    slice = fablib.get_slice(SLICE_NAME)
    nodes = {name: slice.get_node(name) for name in NODE_NAMES}

    print('Checking DeepPrep outputs per subject:')
    for sub, node_name in SUBJECT_NODE_MAP.items():
        node = nodes[node_name]
        stdout, _ = node.execute(
            f'ls {REMOTE_OUTPUT_DIR}/BOLD/sub-{sub}/func/*preproc_bold*.nii.gz 2>/dev/null | wc -l'
        )
        bold_count = stdout.strip()
        stdout2, _ = node.execute(
            f'ls {REMOTE_OUTPUT_DIR}/BOLD/sub-{sub}/func/*confounds_timeseries*.tsv 2>/dev/null | wc -l'
        )
        conf_count = stdout2.strip()
        status = '✅' if int(bold_count) > 0 else '❌'
        print(f'  {status} sub-{sub} @ {node_name}: BOLD={bold_count}  confounds={conf_count}')
except Exception as e:
    print(f'❌ Failed to verify outputs: {e}')

Checking DeepPrep outputs per subject:
2
1
  ✅ sub-001 @ DeepPrepNode1: BOLD=2  confounds=1
2
1
  ✅ sub-002 @ DeepPrepNode2: BOLD=2  confounds=1
0
0
  ❌ sub-003 @ DeepPrepNode3: BOLD=0  confounds=0


## Cell 12 — Copy outputs from nodes to JupyterHub

In [3]:
import os
import subprocess

try:
    slice = fablib.get_slice(SLICE_NAME)
    nodes = {name: slice.get_node(name) for name in NODE_NAMES}

    print('Copying DeepPrep outputs from all 3 nodes to JupyterHub...')
    
    SSH_KEY = '/home/fabric/work/fabric_config/slice_key'
    SSH_CONFIG = '/home/fabric/work/fabric_config/ssh_config'

    for sub, node_name in SUBJECT_NODE_MAP.items():
        if sub == '003':
            print(f'  ⏭️  sub-{sub}: skipping (no func data)')
            continue
            
        node = nodes[node_name]
        ip = node.get_management_ip()
        local_sub_func = os.path.join(LOCAL_OUTPUT_DIR, 'BOLD', f'sub-{sub}', 'func')
        os.makedirs(local_sub_func, exist_ok=True)

        remote_func = f'{REMOTE_OUTPUT_DIR}/BOLD/sub-{sub}/func/'
        
        # scp recursively — IPv6 needs brackets
        cmd = f'scp -r -o StrictHostKeyChecking=no -F {SSH_CONFIG} -i {SSH_KEY} ubuntu@\\[{ip}\\]:{remote_func}* {local_sub_func}/'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
        
        if result.returncode == 0:
            n_files = len(os.listdir(local_sub_func))
            print(f'  ✅ sub-{sub} copied from {node_name} ({n_files} files)')
        else:
            print(f'  ❌ sub-{sub}: {result.stderr[:300]}')

    # Copy participants.tsv from first node
    first_node = nodes[NODE_NAMES[0]]
    ip = first_node.get_management_ip()
    cmd = f'scp -o StrictHostKeyChecking=no -F {SSH_CONFIG} -i {SSH_KEY} ubuntu@\\[{ip}\\]:{REMOTE_BIDS_DIR}/participants.tsv {LOCAL_BIDS_DIR}/'
    subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    print(f'  ✅ participants.tsv copied')
    
    print(f'\n✅ All outputs copied to JupyterHub')
    print(f'  Location: {LOCAL_OUTPUT_DIR}')
except Exception as e:
    print(f'❌ Failed: {e}')


Copying DeepPrep outputs from all 3 nodes to JupyterHub...
  ✅ sub-001 copied from DeepPrepNode1 (22 files)
  ✅ sub-002 copied from DeepPrepNode2 (22 files)
  ⏭️  sub-003: skipping (no func data)
  ✅ participants.tsv copied

✅ All outputs copied to JupyterHub
  Location: /home/fabric/sudmex/deepprep_output


## Cell 13 — Install local post-processing deps (JupyterHub)

In [ ]:
!pip install nilearn numpy pandas matplotlib seaborn --quiet
print('✅ Post-processing dependencies ready')

## Cell 14 — Load confounds and denoise BOLD

In [4]:
import glob
import pandas as pd
import numpy as np
from nilearn import image

DERIVATIVES_DIR = os.path.join(LOCAL_OUTPUT_DIR, 'BOLD')

def load_deepprep_confounds(confounds_path):
    df = pd.read_csv(confounds_path, sep='\t')

    target_cols = [
        'trans_x', 'trans_y', 'trans_z',
        'rot_x',   'rot_y',   'rot_z',
        'trans_x_derivative1', 'trans_y_derivative1', 'trans_z_derivative1',
        'rot_x_derivative1',   'rot_y_derivative1',   'rot_z_derivative1',
        'trans_x_power2', 'trans_y_power2', 'trans_z_power2',
        'rot_x_power2',   'rot_y_power2',   'rot_z_power2',
        'trans_x_derivative1_power2', 'trans_y_derivative1_power2', 'trans_z_derivative1_power2',
        'rot_x_derivative1_power2',   'rot_y_derivative1_power2',   'rot_z_derivative1_power2',
        'white_matter',                    'csf',                    'global_signal',
        'white_matter_derivative1',        'csf_derivative1',        'global_signal_derivative1',
        'white_matter_power2',             'csf_power2',             'global_signal_power2',
        'white_matter_derivative1_power2', 'csf_derivative1_power2', 'global_signal_derivative1_power2',
    ]

    available = [c for c in target_cols if c in df.columns]
    confounds  = df[available].fillna(0).values
    return confounds

# Test on subject 001
sub = '001'
sub_dir        = os.path.join(DERIVATIVES_DIR, f'sub-{sub}', 'func')
bold_files     = sorted(glob.glob(f'{sub_dir}/*preproc_bold*.nii.gz'))
confound_files = sorted(glob.glob(f'{sub_dir}/*confounds_timeseries*.tsv'))

if len(bold_files) == 0:
    print(f'❌ No BOLD files found for sub-{sub} — check that Cell 12 ran successfully')
else:
    confounds_matrix = load_deepprep_confounds(confound_files[0])
    bold_img    = image.load_img(bold_files[0])
    cleaned_img = image.clean_img(
        bold_img,
        confounds=confounds_matrix,
        t_r=2.0,
        low_pass=0.1,
        high_pass=0.01,
        standardize=True,
    )
    print(f'✅ sub-{sub} cleaned')
    print(f'  Raw BOLD shape:     {bold_img.shape}')
    print(f'  Cleaned BOLD shape: {cleaned_img.shape}')

❌ No BOLD files found for sub-001 — check that Cell 12 ran successfully


## Cell 15 — Extract ROI time series

In [ ]:
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker

atlas        = datasets.fetch_atlas_schaefer_2018(n_rois=200, resolution_mm=2)
atlas_img    = atlas.maps
atlas_labels = atlas.labels

masker = NiftiLabelsMasker(
    labels_img=atlas_img,
    standardize=True,
    memory='nilearn_cache',
    verbose=0,
)

time_series = masker.fit_transform(cleaned_img)
print(f'✅ Time series: {time_series.shape[0]} timepoints x {time_series.shape[1]} ROIs')

## Cell 16 — Compute FC matrix (single subject)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn.connectome import ConnectivityMeasure

conn_measure = ConnectivityMeasure(kind='correlation')
fc_matrix    = conn_measure.fit_transform([time_series])[0]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    fc_matrix,
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    xticklabels=False,
    yticklabels=False,
    ax=ax,
)
ax.set_title(f'Functional Connectivity — sub-{sub} (DeepPrep | Schaefer 200)')
plt.tight_layout()
plt.savefig(os.path.join(LOCAL_OUTPUT_DIR, f'sub-{sub}_fc_matrix.png'), dpi=150)
plt.show()
print(f'✅ FC matrix shape: {fc_matrix.shape}')

## Cell 17 — Batch FC extraction (all 3 subjects)

In [ ]:
all_fc = {}

for sub in SUBJECTS:
    sub_dir        = os.path.join(DERIVATIVES_DIR, f'sub-{sub}', 'func')
    bold_list      = sorted(glob.glob(f'{sub_dir}/*preproc_bold*.nii.gz'))
    confounds_list = sorted(glob.glob(f'{sub_dir}/*confounds_timeseries*.tsv'))

    if len(bold_list) == 0 or len(confounds_list) == 0:
        print(f'  ⚠️  sub-{sub}: skipping — missing outputs')
        continue

    try:
        conf_mat   = load_deepprep_confounds(confounds_list[0])
        bold_img   = image.load_img(bold_list[0])
        cleaned    = image.clean_img(
            bold_img,
            confounds=conf_mat,
            t_r=2.0,
            low_pass=0.1,
            high_pass=0.01,
            standardize=True,
        )
        ts         = masker.transform(cleaned)
        fc         = conn_measure.fit_transform([ts])[0]
        out_path   = os.path.join(LOCAL_FC_DIR, f'sub-{sub}_fc.npy')
        np.save(out_path, fc)
        all_fc[sub] = fc
        print(f'  ✅ sub-{sub}: shape={fc.shape}')

    except Exception as e:
        print(f'  ❌ sub-{sub}: ERROR — {e}')

print(f'\nDone. {len(all_fc)}/3 FC matrices saved to {LOCAL_FC_DIR}')

## Cell 18 — Bundle FC matrices + participants.tsv for teammate

In [ ]:
import zipfile
import os

zip_path = os.path.join(LOCAL_BASE, 'classifier_inputs.zip')

with zipfile.ZipFile(zip_path, 'w') as zf:
    for sub in SUBJECTS:
        fc_file = os.path.join(LOCAL_FC_DIR, f'sub-{sub}_fc.npy')
        if os.path.exists(fc_file):
            zf.write(fc_file, arcname=f'fc_matrices/sub-{sub}_fc.npy')
            print(f'  ✅ added sub-{sub}_fc.npy')
        else:
            print(f'  ⚠️  sub-{sub}_fc.npy not found — skipping')

    tsv_path = os.path.join(LOCAL_BIDS_DIR, 'participants.tsv')
    if os.path.exists(tsv_path):
        zf.write(tsv_path, arcname='participants.tsv')
        print(f'  ✅ added participants.tsv')
    else:
        print(f'  ⚠️  participants.tsv not found')

print(f'\n✅ Saved to: {zip_path}')
print('Download this zip and send it to your teammate.')

## Cell 19 — Delete slice when done (saves FABRIC credits)

In [ ]:
# Only run this when you are completely done and all outputs are on JupyterHub

try:
    slice = fablib.get_slice(SLICE_NAME)
    slice.delete()
    print(f'✅ Slice "{SLICE_NAME}" deleted')
except Exception as e:
    print(f'❌ Failed to delete slice: {e}')

## Cell 20 — [NEW] Upload FC extraction script to FABRIC VMsBecause JupyterHub's kernel kept dying from memory pressure when loading preprocessed BOLD files (each ~500MB-1GB), we moved FC extraction onto the FABRIC VMs, which have 64GB RAM each.This cell writes a standalone Python script to each node that:1. Loads preprocessed BOLD from DeepPrep output2. Regresses out motion/WM/CSF confounds3. Applies bandpass filter (0.01-0.1 Hz, resting-state range)4. Extracts ROI time series using Schaefer 200 atlas5. Computes Pearson correlation FC matrix6. Saves as sub-XXX_fc.npy

In [ ]:
# Cell 20: Upload FC extraction script to FABRIC VMsfc_script = '''#!/usr/bin/env python3import os, glob, sysimport numpy as npimport pandas as pdfrom nilearn import image, datasetsfrom nilearn.maskers import NiftiLabelsMaskerfrom nilearn.connectome import ConnectivityMeasuresub_id = sys.argv[1]DERIVATIVES_DIR = "/home/ubuntu/sudmex/deepprep_output/BOLD"OUTPUT_DIR = "/home/ubuntu/sudmex/fc_matrices"os.makedirs(OUTPUT_DIR, exist_ok=True)atlas = datasets.fetch_atlas_schaefer_2018(n_rois=200, resolution_mm=2)masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize=True, verbose=0)conn_measure = ConnectivityMeasure(kind="correlation")def load_confounds(path):    df = pd.read_csv(path, sep="\t")    cols = [c for c in df.columns if any(c.startswith(p) for p in            ["trans_", "rot_", "white_matter", "csf", "global_signal"])]    return df[cols].fillna(0).valuessub_dir = os.path.join(DERIVATIVES_DIR, f"sub-{sub_id}", "func")bold_files = sorted(glob.glob(f"{sub_dir}/*preproc_bold*.nii.gz"))conf_files = sorted(glob.glob(f"{sub_dir}/*confounds_timeseries*.tsv"))if not bold_files:    print(f"NO BOLD FILES in {sub_dir}")    sys.exit(1)print(f"sub-{sub_id}: cleaning BOLD...")confounds = load_confounds(conf_files[0])bold = image.load_img(bold_files[0])cleaned = image.clean_img(bold, confounds=confounds, t_r=2.0,                          low_pass=0.1, high_pass=0.01, standardize=True)print(f"sub-{sub_id}: extracting ROI time series...")ts = masker.fit_transform(cleaned)print(f"sub-{sub_id}: computing FC matrix...")fc = conn_measure.fit_transform([ts])[0]out = os.path.join(OUTPUT_DIR, f"sub-{sub_id}_fc.npy")np.save(out, fc)print(f"DONE sub-{sub_id}: FC saved shape={fc.shape} to {out}")'''# Install nilearn on each node and write the scriptslice = fablib.get_slice(SLICE_NAME)for name in NODE_NAMES:    node = slice.get_node(name)    print(f"{name}: installing nilearn...")    node.execute('pip3 install --quiet --user nilearn scipy scikit-learn')    print(f"{name}: writing FC extraction script...")    node.execute(f"cat > /home/ubuntu/extract_fc.py << 'PYEOF'\n{fc_script}\nPYEOF")print("\n✅ FC extraction scripts deployed to all nodes")

## Cell 21 — [NEW] Run FC extraction on all FABRIC VMs (parallel)Each node extracts FC matrices for its assigned subjects. Uses `/usr/bin/python3` with PATH set so nilearn is found from `~/.local/bin`.Runtime: ~5-10 min per subject (the `image.clean_img` step with bandpass filter is the slow part).

In [ ]:
# Cell 21: Run FC extraction in parallel across all FABRIC VMsimport threadingSUBJECTS_TO_EXTRACT = {    'DeepPrepNode1': ['001'],    'DeepPrepNode2': ['002'],}def extract_fc_on_node(node_name, subs):    node = slice.get_node(node_name)    for sub in subs:        cmd = f'PATH=$PATH:/home/ubuntu/.local/bin /usr/bin/python3 /home/ubuntu/extract_fc.py {sub}'        print(f"[{node_name}] extracting FC for sub-{sub}...")        stdout, stderr = node.execute(cmd)        if "DONE" in stdout:            print(f"[{node_name}] ✅ sub-{sub} complete")        else:            print(f"[{node_name}] ⚠️ sub-{sub}: {stdout[-300:]}")threads = []for node_name, subs in SUBJECTS_TO_EXTRACT.items():    t = threading.Thread(target=extract_fc_on_node, args=(node_name, subs))    threads.append(t)    t.start()for t in threads:    t.join()print("\n✅ All FC extraction complete")

## Cell 22 — [NEW] Download FC matrices from FABRIC VMs to JupyterHubUses `scp` through the FABRIC bastion to pull the `.npy` files (each ~320KB) back to a persistent location on JupyterHub (`/home/fabric/work/`).

In [ ]:
# Cell 22: Download FC matrices + participants.tsv to JupyterHubimport osimport subprocessSSH_KEY = '/home/fabric/work/fabric_config/slice_key'SSH_CONFIG = '/home/fabric/work/fabric_config/ssh_config'# Use /home/fabric/work/ — persists across server restartsOUTPUT_DIR = '/home/fabric/work/classifier_data'os.makedirs(f'{OUTPUT_DIR}/fc_matrices', exist_ok=True)# Map: subject -> node that has itFC_LOCATIONS = {    '001': 'DeepPrepNode1',    '002': 'DeepPrepNode2',}for sub, node_name in FC_LOCATIONS.items():    node = slice.get_node(node_name)    ip = node.get_management_ip()    cmd = f'scp -o StrictHostKeyChecking=no -F {SSH_CONFIG} -i {SSH_KEY} ubuntu@\\[{ip}\\]:/home/ubuntu/sudmex/fc_matrices/sub-{sub}_fc.npy {OUTPUT_DIR}/fc_matrices/'    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)    if result.returncode == 0:        print(f'  ✅ sub-{sub} downloaded from {node_name}')    else:        print(f'  ❌ sub-{sub}: {result.stderr[:200]}')# Also download participants.tsvnode1 = slice.get_node('DeepPrepNode1')ip1 = node1.get_management_ip()cmd = f'scp -o StrictHostKeyChecking=no -F {SSH_CONFIG} -i {SSH_KEY} ubuntu@\\[{ip1}\\]:/home/ubuntu/sudmex/bids/participants.tsv {OUTPUT_DIR}/'subprocess.run(cmd, shell=True, capture_output=True, text=True)print('  ✅ participants.tsv downloaded')print(f'\n✅ All files in: {OUTPUT_DIR}')print(f'Contents: {os.listdir(f"{OUTPUT_DIR}/fc_matrices")}')

## Cell 23 — [NEW] Batch processing for additional subjectsTo get enough subjects for classification (need at least 3 per group), we process more subjects after the initial run. This cell downloads more BIDS data + runs DeepPrep + extracts FC for additional subjects in parallel across the 3 VMs.Each node handles 2 new subjects sequentially. Total wall time: ~90 minutes for 6 additional subjects (3 per group).

In [ ]:
# Cell 23: Process additional subjects for classificationimport threadingimport timeNEW_SUBJECTS = {    'DeepPrepNode1': ['004', '005'],  # group 1 (cocaine user)    'DeepPrepNode2': ['006', '008'],  # group 2 (control)    'DeepPrepNode3': ['007', '009'],  # mixed}def process_subjects_on_node(node_name, subjects):    node = slice.get_node(node_name)    print(f"\n[{node_name}] Processing: {subjects}")    # Ensure FS license is present    license_escaped = FS_LICENSE_CONTENT.replace('"', '\\"')    node.execute(f'echo "{license_escaped}" > /home/ubuntu/sudmex/fs_license.txt')    for sub in subjects:        print(f"[{node_name}] >>> Starting sub-{sub}")        # Download BIDS data for this specific subject        download_script = f'''import openneuroopenneuro.download(    dataset="ds003346",    target_dir="/home/ubuntu/sudmex/bids",    include=["sub-{sub}", "dataset_description.json", "participants.tsv", "participants.json", "README.md"],)print("Download done")'''        node.execute(f"cat > /home/ubuntu/dl_sub_{sub}.py << 'PYEOF'\n{download_script}\nPYEOF")        node.execute(f'sudo python3 /home/ubuntu/dl_sub_{sub}.py')        print(f"[{node_name}] sub-{sub}: download done")        # Run DeepPrep        deepprep_cmd = (            f'sudo docker run --rm '            f'-v /home/ubuntu/sudmex/bids:/input '            f'-v /home/ubuntu/sudmex/deepprep_output:/output '            f'-v /home/ubuntu/sudmex/fs_license.txt:/fs_license.txt '            f'pbfslab/deepprep:25.1.0 '            f'/input /output participant '            f'--bold_task_type rest '            f'--fs_license_file /fs_license.txt '            f'--participant_label sub-{sub} '  # `sub-` prefix avoids Nextflow octal parsing            f'--bold_confounds '            f'--bold_volume_space MNI152NLin2009cAsym '            f'--bold_surface_spaces None '            f'--bold_skip_frame 2 '            f'--device cpu --cpus 32 --memory 64 --ignore_error'        )        log_path = f'/home/ubuntu/sudmex/deepprep_sub-{sub}.log'        bg_cmd = f'nohup bash -c "{deepprep_cmd} > {log_path} 2>&1" &'        node.execute(bg_cmd)        # Poll until DeepPrep completes        start = time.time()        while True:            running, _ = node.execute('pgrep -af "docker run" | grep -v pgrep | wc -l')            if int(running.strip()) == 0:                elapsed = int(time.time() - start)                print(f"[{node_name}] sub-{sub}: DeepPrep done in {elapsed//60}m")                break            time.sleep(120)        # Extract FC        stdout, _ = node.execute(            f'PATH=$PATH:/home/ubuntu/.local/bin /usr/bin/python3 /home/ubuntu/extract_fc.py {sub}'        )        if "DONE" in stdout:            print(f"[{node_name}] ✅ sub-{sub} FC extracted")threads = []for node_name, subs in NEW_SUBJECTS.items():    t = threading.Thread(target=process_subjects_on_node, args=(node_name, subs))    threads.append(t)    t.start()for t in threads:    t.join()print("\n🎉 All additional subjects processed")

## Cell 24 — [NEW] Bundle final classifier inputsCreates `classifier_inputs.zip` containing all FC matrices + participants.tsv. Download this from JupyterHub's `work/` folder to your local machine and run the ML pipeline against it.

In [ ]:
# Cell 24: Bundle everything for the ML pipelineimport osimport zipfileOUTPUT_DIR = '/home/fabric/work/classifier_data'# First download ALL FC matrices from ALL nodesFC_LOCATIONS = {    '001': 'DeepPrepNode1',    '005': 'DeepPrepNode1',    '002': 'DeepPrepNode2',    '008': 'DeepPrepNode2',    # Add more as processed}SSH_KEY = '/home/fabric/work/fabric_config/slice_key'SSH_CONFIG = '/home/fabric/work/fabric_config/ssh_config'os.makedirs(f'{OUTPUT_DIR}/fc_matrices', exist_ok=True)import subprocessfor sub, node_name in FC_LOCATIONS.items():    node = slice.get_node(node_name)    ip = node.get_management_ip()    cmd = f'scp -o StrictHostKeyChecking=no -F {SSH_CONFIG} -i {SSH_KEY} ubuntu@\\[{ip}\\]:/home/ubuntu/sudmex/fc_matrices/sub-{sub}_fc.npy {OUTPUT_DIR}/fc_matrices/'    subprocess.run(cmd, shell=True, capture_output=True, text=True)# Create the zipzip_path = '/home/fabric/work/classifier_inputs.zip'with zipfile.ZipFile(zip_path, 'w') as zf:    for fname in sorted(os.listdir(f'{OUTPUT_DIR}/fc_matrices')):        fpath = f'{OUTPUT_DIR}/fc_matrices/{fname}'        zf.write(fpath, arcname=f'fc_matrices/{fname}')        print(f'  ✅ added {fname}')    tsv = f'{OUTPUT_DIR}/participants.tsv'    if os.path.exists(tsv):        zf.write(tsv, arcname='participants.tsv')        print(f'  ✅ added participants.tsv')print(f'\n✅ Saved: {zip_path}')print('Download this from JupyterHub\'s work/ folder and run the ML pipeline locally.')

## Running the ML pipeline locallyOnce you download `classifier_inputs.zip`, on your local machine:```bashcd ~/CS4001/FINAL-PROJECT/unzip classifier_inputs.zip -d classifier_data/cd ml_pipelinepython3 -m venv venvsource venv/bin/activatepip install numpy pandas scikit-learn matplotlib seabornpython3 main.py \    --fc-dir ~/CS4001/FINAL-PROJECT/classifier_data/fc_matrices \    --participants ~/CS4001/FINAL-PROJECT/classifier_data/participants.tsv \    --tags cocaine \    --output-dir ./results \    --cv-splits 2```## Our ResultsWith 4 subjects (2 per class), cocaine users vs healthy controls:| Model | Balanced Accuracy | ROC-AUC ||-------|-------------------|---------|| LogisticRegression (L2) | **0.75** | 0.75 || SVM Linear | 0.75 | 0.25 || Random Forest | 0.75 | 0.50 || LogisticRegression (L1) | 0.50 | 0.50 || SVM RBF | 0.50 | 0.50 |Best model correctly classified 3 out of 4 subjects. With n=4 this is extremely noisy but demonstrates the pipeline works end-to-end.